In [ ]:
import time
from google import genai
from tqdm import tqdm
from dotenv import load_dotenv
import os
load_dotenv()

from google.cloud import aiplatform
from google.cloud.aiplatform.metadata import context
from google.cloud.aiplatform.metadata import utils as metadata_utils
from google.genai import types
import vertexai 
from google.cloud import storage

# For data handling.
import json
import jsonlines
import pandas as pd
from sklearn.model_selection import train_test_split



In [2]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 
BUCKET = "sales_recommender_dev_bucket"
DATASET_NAME = "labeled_df_relevance.csv"

# Initialize Vertex AI and GenAI clients
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(PROJECT_ID)

In [3]:
prompt = '''

    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.

        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.

        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. 
        Examples of specialized projects are: 
            * Hospital and health services
            * Churches
            * Commercial real estate
            * Large residential apartments/dormitories
            * University/College buildings
            * Auditoriums
            * Senior living homes
        Examples of non-specialized projects with very low priority are:
            * One-time projects
            * Small residential projects
            * Golf courses

        4. **Identify Building Type**: Identify building type, including interior complexity and specialized work. 

        5. **Identify Locations and Distance:** Identify the project location and distance from nearest branch. Consider if a branch is too far away from a location. 
        Urban areas should have closer branches, while rural areas can have branches further away.

        6. **Identify Associated Brands:** Identify associated brands to the product. Associated brands include: 
            * Armstrong Ceilings 
            * Sto 
            * Dryvit

        7. **Identify Available Plans:** Identify if the project has detailed and available plans and specs.

        8. **Classify Projects:**
            a. Prioritize projects based on how relevant the inputs are to the search terms.
            b. Next, prioritize projects based on the project type, as specified in the previous steps. Deprioritize non-specialized projects. 
            c. Next, prioritize building types based on how complex the interior work is, as specified in the previous steps. Deprioritize projects with little interior work.
            d. Next, prioritize projects that have reasonable distance to the nearest branch, as specified in the previous steps. Deprioritize projects that are too far away from a branch.
            e. Next, prioritize projects that have associated brands, as specified in the previous steps. Lack of associated brands will not lower the priority.
            f. Next, increase priority if the project has detailed plans and specs. Lack of plans and specs will not lower the priority. 
            g. When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        8. **Estimate Relevancy:** Estimate the relevancy of each project based on the above factors and total dollar amount.

        9. Respond in valid JSON: Return only in valid JSON format. The JSON should contain the classification of the project as very high, high, moderate, low, or not relevant. 
        ''' 

In [4]:
# Read df 
df = pd.read_csv(DATASET_NAME)

# Preprocess 
relevance_col_name = "Relevance"
df.dropna(subset=[relevance_col_name], inplace=True)
df.drop(columns = ['Query'], inplace = True)

# Split data
X = df.drop(columns=[relevance_col_name])
y = df[[relevance_col_name]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Combine ProjectID with y_train and y_test
y_train = pd.concat([pd.DataFrame(X_train['ProjectID']), y_train], axis = 1)
y_test = pd.concat([pd.DataFrame(X_test['ProjectID']), y_test], axis = 1)

In [76]:
# Format row contents and prompts as a list of parts
def format_contents_as_part(row, prompt, boolean_filters, examples: bool=True): 

    product = row['Search']
    input_row = row.drop('Search')
    search_terms = [filter['Query'] for filter in boolean_filters if filter['Filter'] == product][0]

    row_json_str = input_row.to_json()
    cc_project_json_record = json.loads(row_json_str)

    parts = [
            {"text": prompt},
            {"text": "**Input Data:** \n "},
            {"text": "**Search:** \n " + product},
            {"text": "**Project Data:** \n " + json.dumps(cc_project_json_record)},
            {"text": "**Search Terms:** \n " + search_terms}]

    if examples: 
        examples_txt = f'''**Example Output:** \n
      [
        {{"ProjectID": 1837563703, "Relevance": "Not Relevant"}}
        {{"ProjectID": 1837563704, "Relevance": "Low"}}
        {{"ProjectID": 1006193701, "Relevance": "Moderate"}}
        {{"ProjectID": 1837563705, "Relevance": "High"}}
        {{"ProjectID": 1006193702, "Relevance": "Very High"}}
      ]
        '''
        parts.append({"text": examples_txt})
        
    return parts

In [77]:
# Prepare JSONL training data for tuning 
import pandas as pd 
import json 
boolean_filters = pd.read_csv(f"gs://{BUCKET}/data/boolean_filters_latest.csv")
boolean_filters = json.loads(boolean_filters.to_json(orient='records'))

lines = []
for index, row in X_train.iterrows():

    parts = format_contents_as_part(row, prompt, boolean_filters)

    # Get the corresponding output from y_train
    output_json = y_train.loc[index].to_json()

    # Create the content row
    content_row = {}
    content_row["contents"] = []
    content_row["contents"].append({"role": "user", "parts": parts})
    content_row["contents"].append({"role": "model", "parts": [{"text": output_json}]})

    # Append the content row to the lines list
    lines.append(content_row)

In [78]:
lines[0]['contents']

[{'role': 'user',
  'parts': [{'text': '\n\n    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.\n\n        **Instructions:**\n\n        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.\n\n        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.\n\n        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. \n        Examples of specialized projects are: \n            * Hospital and health services\n            * Churches\n            * Commercial real estate\n            * Large residential apartments/dormitories\n            * University/College buildings\n            * Auditoriums\n        

In [79]:
# Write the lines to a JSONL file
output_file = "training_data_for_tuning_job.jsonl"
with open(output_file, 'w') as f:
    for line in lines:
        f.write(json.dumps(line) + '\n')

# Upload the file to GCS
bucket = storage_client.bucket(BUCKET)
blob = bucket.blob(f'data/tuning/{output_file}')
blob.upload_from_filename(output_file)

In [83]:
# Format testing data for prediction
X_test_formatted = [] 

for _, row in X_test.iterrows(): 
    
    parts = format_contents_as_part(row, prompt, boolean_filters)

    # Create the content row
    content_row = {"contents": []}
    for part in parts:
        content_row["contents"].append(part['text'])

    # Append the content row to the lines list
    X_test_formatted.append(content_row)

X_test_formatted[0]['contents']

['\n\n    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.\n\n        **Instructions:**\n\n        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect project is provided in **Project Data**.\n\n        2. **Utilize Search Terms:** Identify relevant products, materials, and phrases. The search terms are provided in the **Search Terms** section.\n\n        3. **Consider Project Types:** Identify project type. Determine if the project is specialized and has high opportunity for work and visibility. \n        Examples of specialized projects are: \n            * Hospital and health services\n            * Churches\n            * Commercial real estate\n            * Large residential apartments/dormitories\n            * University/College buildings\n            * Auditoriums\n            * Senior living homes\n        Exa

In [ ]:
# Tune the model 
base_model = "gemini-2.0-flash-001"
tuned_model_name = "tuned-model"

from vertexai.tuning import sft 

sft_tuning_job = sft.train(
    source_model = base_model,
    train_dataset = f"gs://{BUCKET}/data/tuning/{output_file}",
    tuned_model_display_name = tuned_model_name, 
    epochs = 50
)

In [ ]:
# tuning_job = client.tunings.get(name=sft_tuning_job.resource_name)
tuning_job = client.tunings.get(name ='projects/195063057478/locations/us-central1/tuningJobs/3935798920705212416')
tuned_model_endpoint = tuning_job.tuned_model.endpoint
tuned_model_endpoint

'projects/195063057478/locations/us-central1/endpoints/656581065107832832'

In [82]:
# Get tuned and based
from vertexai.generative_models import GenerativeModel, GenerationConfig 

response_schema = {
        "type": "OBJECT",
        "properties": {
            "ProjectID": {"type": "INTEGER"},
            "Relevance": {"type": "STRING"}
        }
}

config = GenerationConfig(response_schema=response_schema, response_mime_type="application/json")

MODEL_ID = "gemini-2.0-flash-001"

tuned_model = GenerativeModel(tuned_model_endpoint)
base_model = GenerativeModel(MODEL_ID)

tuned_predictions = [] 
base_predictions = []
for item in tqdm(X_test_formatted): 
    tuned_response = tuned_model.generate_content(contents = item["contents"], generation_config = config)
    tuned_predictions.append(tuned_response.text)

    base_response = base_model.generate_content(contents = item["contents"], generation_config = config)
    base_predictions.append(base_response.text)

100%|██████████| 30/30 [00:52<00:00,  1.73s/it]


In [81]:
def strip_json_response(response):
    output = response.replace("```json", "").replace("```", "").replace("[", "").replace("]", "").replace("\n","").strip() 
    output = json.loads(output)
    return output['Relevance']

cleaned_tuned_predictions = [strip_json_response(pred) for pred in tuned_predictions]
cleaned_based_predictions = [strip_json_response(pred) for pred in base_predictions]

predictions_df = y_test.copy() 
predictions_df['Tuned Relevance'] = cleaned_tuned_predictions
predictions_df['Base Relevance'] = cleaned_based_predictions
predictions_df.head()

,ProjectID,Relevance,Tuned Relevance,Base Relevance
109,1004206072,High,High,Moderate
14,1002057138,Very High,Very High,High
107,1004182711,Very High,Very High,Not Relevant
122,1004311228,Very High,Very High,High
62,1002464636,High,Very High,Moderate


In [66]:

from sklearn.preprocessing import OrdinalEncoder 
relevance_columns = predictions_df.filter(regex='Relevance').columns

# Encode the relevance columns
labels = ['not relevant', 'low', 'moderate', 'high', 'very high', 'very low']
encoder = OrdinalEncoder(categories=[labels])

for col in relevance_columns: 
    predictions_df[col] = predictions_df[col].astype(str).str.lower()
    predictions_df[col] = encoder.fit_transform(predictions_df[[col]])



In [67]:
from sklearn.metrics import f1_score

print(f"F1 Score for {col}: ", f1_score(predictions_df['Relevance'], predictions_df["Base Relevance"], average='weighted'))
print(f"F1 Score for Tuned Predictions: ", f1_score(predictions_df['Relevance'], predictions_df["Tuned Relevance"], average='weighted'))

F1 Score for Base Relevance:  0.2713800904977376
F1 Score for Tuned Predictions:  0.3568322981366459
